In [1]:
# /// script
# dependencies = [
#   "geopandas",
#   "lonboard",
#   "pyarrow",
#   "ipywidgets",
#   "numpy",
# ]
# ///

import geopandas as gpd
import numpy as np
import ipywidgets as widgets
from lonboard import Map, ScatterplotLayer, PolygonLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap

# 1. Download Live Real-Time Earthquakes (USGS 7-Day Feed)
usgs_url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_week.geojson"
land_url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_land.geojson"

eq_gdf = gpd.read_file(usgs_url)
land_gdf = gpd.read_file(land_url)

# Extract depth from 3D Point coordinates (Z-coordinate)
eq_gdf["depth_km"] = eq_gdf.geometry.z.fillna(0.0)

# Filter out null or negative magnitudes
eq_gdf = eq_gdf[eq_gdf["mag"] > 0].copy()

# Reset index to ensure clean GeoArrow conversion without index artifacts
eq_gdf = eq_gdf.reset_index(drop=True)

# Select clean metadata for inspection
eq_gdf["Location"] = eq_gdf["place"].fillna("Unknown Location")
eq_gdf["Magnitude"] = eq_gdf["mag"]
eq_gdf["Depth (km)"] = eq_gdf["depth_km"].round(1)

# 2. Assign Color Ramping as a 2D NumPy Array
def assign_depth_color(depth):
    if depth < 30:
        return [255, 170, 0, 220]    # Shallow - Bright Yellow-Orange
    elif depth < 70:
        return [255, 50, 50, 220]    # Intermediate - Red
    else:
        return [200, 0, 255, 230]    # Deep - Bright Magenta-Purple

# Convert list of colors to a contiguous uint8 2D numpy array (shape: N x 4)
color_array = np.array([assign_depth_color(d) for d in eq_gdf["Depth (km)"]], dtype=np.uint8)

# 3. Context Layer (Dark Landmasses)
land_layer = PolygonLayer.from_geopandas(
    land_gdf,
    get_fill_color=[20, 24, 32, 255],
    get_line_color=[45, 50, 60, 255],
    get_line_width=300,
)

# 4. Earthquake Epicenter Scatterplot Layer
# Point radius scales dynamically with earthquake magnitude
eq_layer = ScatterplotLayer.from_geopandas(
    eq_gdf[["geometry", "Location", "Magnitude", "Depth (km)"]],
    get_fill_color=color_array,             # Fixed: Passing 2D NumPy Array
    get_radius=eq_gdf["Magnitude"] * 3500,  # Scaled by magnitude
    radius_min_pixels=2,
    radius_max_pixels=30,
    opacity=0.8,
    pickable=True,
    auto_highlight=True,
    highlight_color=[0, 255, 200, 255]     # Electric Cyan highlight on selection
)

# 5. Construct Map Viewport
m = Map(
    layers=[land_layer, eq_layer],
    basemap=MaplibreBasemap(style=CartoStyle.DarkMatterNoLabels),
    view_state={
        "longitude": 0.0,
        "latitude": 20.0,
        "zoom": 1.8,
    },
    picking_radius=10
)

# 6. Interactive Inspection Card
info_box = widgets.HTML(
    value="<div style='background-color: #1e222b; padding: 12px; border-radius: 6px; color: white;'>"
          "<b>⚡ Seismic Activity Inspector:</b> Hover or click on any earthquake epicenter to view magnitude and hypocenter depth."
          "</div>"
)

def on_eq_select(change):
    idx = change["new"]
    if idx is not None and idx >= 0:
        feature = eq_gdf.iloc[idx]
        info_box.value = f"""
        <div style="background-color: #1e222b; padding: 12px; border-radius: 6px; border-left: 5px solid #FF3232; color: white;">
            <b style="color: #FF3232; font-size: 16px;">⚡ M {feature['Magnitude']} - {feature['Location']}</b><br/>
            <span><b>Hypocenter Depth:</b> {feature['Depth (km)']} km</span>
        </div>
        """

eq_layer.observe(on_eq_select, names="selected_index")

# Legend & Dashboard Layout
legend_html = widgets.HTML("""
<div style="color: white; margin-bottom: 8px;">
    <b>Depth Legend:</b> 
    <span style="color: #FFAA00;">■ Shallow (&lt;30km)</span> &nbsp;|&nbsp;
    <span style="color: #FF3232;">■ Intermediate (30-70km)</span> &nbsp;|&nbsp;
    <span style="color: #C800FF;">■ Deep (&gt;70km)</span>
</div>
""")

widgets.VBox([
    widgets.HTML("<h2>⚡ Live Global Seismic Activity & Earthquake Depth (USGS 7-Day Feed)</h2>"),
    legend_html,
    info_box,
    m
])

/Users/aryan/sw_engg/opensource/lonboard-examples/lb/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(
